# LJ Dev Commerce - Phase 4

# I. Customer ETL

### Source System
CRM (Customer Relationship Management)

### Target Table
commerce.customer

### Purpose
Load customer data from the CRM export into the centralized PostgreSQL database after profiling, cleaning, validating, and transforming the data.

In [2]:
import pandas as pd
import re

## 1. Load the CRM Source Data

The CRM export will be loaded into a Pandas DataFrame for data profiling, cleaning, transformation, and validation before being loaded into the centralized PostgreSQL database.

In [3]:
df = pd.read_excel("../data/crm_customers_export_2026_08_01.xlsx")

In [4]:
df.head()

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
0,C001,Jeff Sanchez,jeff@email.com,+971559522199,Al Jafiliya,Dubai,UAE,Active,2026-07-01,crm_admin,2026-07-15,crm_admin
1,C002,Maria Santos,maria@email.com,+639171234567,Makati,Manila,Philippines,Active,2026/07/02,crm_admin,2026/07/10,crm_admin
2,C003,AHMED ALI,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,01-07-2026,crm_admin,15-07-2026,crm_admin
3,C004,John Cruz,john@email.com,97150ABC123,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin
4,C005,Anne Reyes,anne@email.com,+639181112223,NaN,Cebu,Philippines,Active,2026-07-04,crm_admin,2026-07-12,crm_admin


## 2. Python Data Profiling

In [5]:
df.shape

(16, 12)

### 2.1 Dataset Size

**Result:** The source dataset contains 16 rows and 12 columns.

**Finding:** The CRM export contains 16 customer records across 12 source columns.

**ETL Decision:** No action required at this stage. The dataset size will be used as a baseline for later validation to ensure records are not unintentionally lost or duplicated during ETL.

In [6]:
df.isnull().sum()

Cust ID          0
Full Name        0
Email Address    3
Mobile           1
Address          1
City             0
Country          0
Status           0
Created Date     0
Created By       0
Updated Date     0
Updated By       0
dtype: int64

### 2.2 Missing Values

**Result:** Missing values were identified in the following columns:

- Email Address: 3 missing values
- Mobile: 1 missing value
- Address: 1 missing value

**Finding:** Missing values were found only in optional fields according to the Customer Data Dictionary.

**ETL Decision:** Preserve the missing values as NULL. No imputation will be performed because these fields are optional and there is currently no business rule requiring replacement values.

In [7]:
df["Cust ID"].duplicated().sum()

np.int64(1)

### 2.3 Duplicate Customer IDs

**Result:** Python detected 1 duplicate Customer ID occurrence.

**Finding:** The Customer ID uniqueness rule is violated. At least one Customer ID appears more than once in the source data.

**ETL Decision:** Investigate the duplicated Customer ID and determine whether the records represent a true duplicate or separate records incorrectly sharing the same ID. Do not remove any record until the duplicate is investigated.

In [8]:
df[df["Cust ID"].duplicated(keep=False)]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
2,C003,AHMED ALI,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,01-07-2026,crm_admin,15-07-2026,crm_admin
10,C003,AHMED ALI,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,01-07-2026,crm_admin,15-07-2026,crm_admin


### 2.3.1 Duplicate Investigation

**Result:** Customer ID `C003` appears twice in the source data.

**Finding:** The two `C003` records contain identical customer information, indicating an exact duplicate record rather than two separate customers sharing the same Customer ID.

**ETL Decision:** Keep one `C003` record and remove the exact duplicate during the cleaning stage. The final dataset must contain only one record per Customer ID.

In [9]:
df.dtypes

Cust ID          object
Full Name        object
Email Address    object
Mobile           object
Address          object
City             object
Country          object
Status           object
Created Date     object
Created By       object
Updated Date     object
Updated By       object
dtype: object

### 2.4 Data Types

**Result:** All 12 source columns are currently interpreted by Pandas as `object`.

**Finding:** The text-based columns are appropriately represented as general object/text values. However, Created Date and Updated Date are also interpreted as `object` rather than datetime values. This is consistent with the inconsistent date formats observed in the source data.

**ETL Decision:** Convert Created Date and Updated Date to standardized datetime values during the cleaning/transformation stage. Text-based columns will remain as text unless further profiling identifies a specific data-type requirement.

In [10]:
df["Status"].unique()

array(['Active', 'Inactive', 'active', 'ACTIVE'], dtype=object)

### 2.5 Status Values

**Result:** Python identified four distinct Status values: `Active`, `Inactive`, `active`, and `ACTIVE`.

**Finding:** The source contains inconsistent capitalization for the same business values. No unexpected status category was identified.

**ETL Decision:** Standardize Status values to `Active` and `Inactive` during the cleaning stage.


In [11]:
df["Country"].unique()

array(['UAE', 'Philippines', 'Singapore', 'Korea'], dtype=object)

### 2.6 Country Values

**Result:** Python identified four distinct Country values: `UAE`, `Philippines`, `Singapore`, and `Korea`.

**Finding:** The Country values are consistently formatted, with no obvious capitalization or naming inconsistencies identified during profiling.

**ETL Decision:** No Country transformation is required based on the current profiling results. Preserve the source values unless a broader business standard requires different country naming.

In [12]:
df["Full Name"].tolist()

[' Jeff Sanchez ',
 'Maria Santos',
 'AHMED ALI',
 'John Cruz',
 'Anne Reyes',
 'Peter Lim',
 'Lara Dizon',
 'Kevin Tan',
 ' Fatima Noor',
 'Chris Lee',
 'AHMED ALI',
 'mary ann dela cruz',
 'Robert Ong',
 'Grace Yu',
 'Nina Lopez',
 ' Omar Hassan ']

In [13]:
df["Full Name"].str.strip().ne(df["Full Name"]).sum()

np.int64(3)

### 2.7.1 Full Name — Whitespace

**Result:** Python detected 3 Full Name records with leading or trailing whitespace.

**Finding:** Some customer names contain unnecessary spaces at the beginning or end of the value.

**ETL Decision:** Remove leading and trailing whitespace during the cleaning stage.

In [14]:
df["Full Name"].str.strip().str.title().ne(df["Full Name"].str.strip()).sum()

np.int64(3)

### 2.7.2 Full Name — Capitalization

**Result:** Python detected 3 Full Name records with inconsistent capitalization.

**Finding:** Inconsistent capitalization was identified, including uppercase names and lowercase names.

**ETL Decision:** Standardize customer name capitalization during the cleaning stage, subject to the defined business formatting rule.

In [15]:
df["Email Address"].dropna().tolist()

['jeff@email.com',
 'maria@email.com',
 'john@email.com',
 'anne@email.com',
 'peter@email.com',
 'lara@email.com',
 'kevin@email.com',
 'fatima@email.com',
 'chris@email.com',
 'mary@email.com',
 'robert@email.com',
 'grace@email.com',
 'omar@email.com']

In [16]:
df["Email Address"].dropna().str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$").sum()

np.int64(13)

### 2.8 Email Address Quality

**Result:** The source contains 13 non-missing email addresses. All 13 passed the basic email format validation.

**Finding:** No invalid email formats were detected among the provided email addresses. Three email values are missing, which is allowed because Email Address is an optional field.

**ETL Decision:** Preserve the 3 missing email values as NULL. No email-format correction is required based on the current profiling results.

In [17]:
df["Mobile"].dropna().tolist()

['+971559522199',
 '+639171234567',
 '+971501112223',
 '97150ABC123',
 '+639181112223',
 '+6591234567',
 '+971556667778',
 '+971509998887',
 '+821012345678',
 '+971501112223',
 '+639191234567',
 '+971558887776',
 '+971551234567',
 '+639177778888',
 '+971554445556']

In [18]:
df["Mobile"].dropna().str.match(r"^\+?[0-9]+$").sum()

np.int64(14)

### 2.9.1 Phone Number — Basic Format

**Result:** The source contains 15 non-missing phone numbers. 14 passed the basic character validation, while 1 failed.

**Finding:** One phone number contains alphabetic characters (`+97150ABC123`) and does not meet the basic phone-number format requirement.

**ETL Decision:** Flag the invalid phone number for review or correction. Do not automatically invent or modify the number. Valid phone numbers will be preserved.

In [19]:
df[df["Mobile"].notna() & ~df["Mobile"].str.match(r"^\+?[0-9]+$", na=False)]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
3,C004,John Cruz,john@email.com,97150ABC123,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin


### 2.9.2 Phone Number — Invalid Record Investigation

**Result:** Customer ID `C004` (John Cruz) contains the invalid phone number `97150ABC123`.

**Finding:** The phone number contains alphabetic characters and fails the basic phone-format validation.

**ETL Decision:** Do not invent or modify the phone number. Flag the record for CRM/business-owner review. If the correct phone number is provided, update it; otherwise, load the phone value as NULL rather than loading invalid data.

In [20]:
df["Address"].dropna().tolist()

['Al Jafiliya',
 'Makati',
 'Al Nahda',
 'Business Bay',
 'Orchard',
 'Quezon City',
 'Marina',
 'Deira',
 'Gangnam',
 'Al Nahda',
 'Pasig',
 'JVC',
 'Karama',
 'Davao',
 'Sharjah']

In [21]:
df["Address"].dropna().str.strip().ne(df["Address"].dropna()).sum()

np.int64(0)

### 2.10.1 Address — Whitespace

**Result:** Python detected 0 Address records with leading or trailing whitespace.

**Finding:** No leading or trailing whitespace issues were identified in the provided Address values.

**ETL Decision:** No Address whitespace cleaning is required based on the current profiling results.

In [22]:
df["City"].unique()

array(['Dubai', 'Manila', 'Cebu', 'Singapore', 'Abu Dhabi', 'Seoul',
       'manila', 'Davao', 'Sharjah'], dtype=object)

### 2.11 City Values

**Result:** Python identified 9 distinct City values. The values `Manila` and `manila` were both found.

**Finding:** An inconsistent capitalization was identified for the city value Manila.

**ETL Decision:** Standardize City capitalization during the cleaning stage according to the agreed business formatting rule.

In [23]:
df["Created Date"].tolist()

['2026-07-01',
 '2026/07/02',
 '01-07-2026',
 '2026-07-03',
 '2026-07-04',
 '2026-07-05',
 '2026-07-06',
 '2026-07-07',
 '2026-07-08',
 '2026-07-09',
 '01-07-2026',
 '2026-07-10',
 '2026-07-11',
 '2026-07-12',
 '2026-07-13',
 '2026-07-14']

### 2.12 Created Date Quality

**Result:** All 16 Created Date values are populated, but multiple date formats were identified, including `YYYY-MM-DD`, `YYYY/MM/DD`, and `DD-MM-YYYY`.

**Finding:** Created Date values are not stored in a consistent source format. The value `01-07-2026` is potentially ambiguous and should be interpreted according to the source system's date convention.

**ETL Decision:** Standardize Created Date to a consistent datetime format during the transformation stage. Confirm the source system's date convention before converting ambiguous values.

In [24]:
pd.to_datetime(df["Created Date"], errors="coerce")

0    2026-07-01
1           NaT
2           NaT
3    2026-07-03
4    2026-07-04
5    2026-07-05
6    2026-07-06
7    2026-07-07
8    2026-07-08
9    2026-07-09
10          NaT
11   2026-07-10
12   2026-07-11
13   2026-07-12
14   2026-07-13
15   2026-07-14
Name: Created Date, dtype: datetime64[ns]

In [25]:
df.loc[pd.to_datetime(df["Created Date"], errors="coerce").isna(), "Created Date"]

1     2026/07/02
2     01-07-2026
10    01-07-2026
Name: Created Date, dtype: object

### 2.12.1 Created Date — Parsing Check

**Result:** Automatic datetime parsing produced `NaT` for 3 records: `2026/07/02` and two occurrences of `01-07-2026`.

**Finding:** Three Created Date values could not be automatically parsed using the current parsing approach because they use different date formats from the majority of the source data. The values appear to represent valid dates but require explicit format handling.

**ETL Decision:** Do not discard these values. Determine the source date convention and apply explicit date-format conversion during the cleaning/transformation stage. Validate the converted dates afterward.

In [26]:
df["Updated Date"].tolist()

['2026-07-15',
 '2026/07/10',
 '15-07-2026',
 '2026-07-11',
 '2026-07-12',
 '2026-07-13',
 '2026-07-14',
 '2026-07-15',
 '2026-07-15',
 '2026-07-15',
 '15-07-2026',
 '2026-07-15',
 '2026-07-15',
 '2026-07-15',
 '2026-07-15',
 '2026-07-15']

### 2.13 Updated Date Quality

**Result:** All 16 Updated Date values are populated, but multiple date formats were identified, including `YYYY-MM-DD`, `YYYY/MM/DD`, and `DD-MM-YYYY`.

**Finding:** Updated Date values are not stored in a consistent source format. The source contains both slash-separated and hyphen-separated date formats.

**ETL Decision:** Standardize Updated Date to a consistent datetime format during the transformation stage. Confirm the source date convention before converting ambiguous values.

In [27]:
pd.to_datetime(df["Updated Date"], errors="coerce")

0    2026-07-15
1           NaT
2           NaT
3    2026-07-11
4    2026-07-12
5    2026-07-13
6    2026-07-14
7    2026-07-15
8    2026-07-15
9    2026-07-15
10          NaT
11   2026-07-15
12   2026-07-15
13   2026-07-15
14   2026-07-15
15   2026-07-15
Name: Updated Date, dtype: datetime64[ns]

### 2.13.1 Updated Date — Parsing Check

**Result:** Automatic datetime parsing produced `NaT` for 3 records: `2026/07/10` and two occurrences of `15-07-2026`.

**Finding:** Three Updated Date values could not be automatically parsed using the current parsing approach because they use different date formats from the majority of the source data. The values appear to represent valid dates but require explicit format handling.

**ETL Decision:** Do not discard these values. Determine the source date convention and apply explicit date-format conversion during the cleaning/transformation stage. Validate the converted dates afterward.

In [28]:
df["Created By"].str.strip().ne(df["Created By"]).sum()

np.int64(0)

### 2.14 Created By — Whitespace

**Result:** Python detected 0 `Created By` records with leading or trailing whitespace.

**Finding:** No leading or trailing whitespace issues were identified.

**ETL Decision:** No `Created By` whitespace cleaning is required based on the current profiling results.

In [29]:
df["Updated By"].str.strip().ne(df["Updated By"]).sum()

np.int64(0)

### 2.15 Updated By — Whitespace

**Result:** Python detected 0 `Updated By` records with leading or trailing whitespace.

**Finding:** No leading or trailing whitespace issues were identified.

**ETL Decision:** No `Updated By` whitespace cleaning is required based on the current profiling results.

### PROFILING SUMMARY (FIRST STAGE)
| Area                     | Result                                     |
| ------------------------ | ------------------------------------------ |
| Dataset size             | 16 rows × 12 columns                       |
| Missing values           | Email 3, Mobile 1, Address 1               |
| Duplicate Customer IDs   | 1 duplicate occurrence                     |
| Duplicate investigation  | C003 confirmed as exact duplicate          |
| Data types               | All `object`; dates need conversion        |
| Status                   | Inconsistent capitalization                |
| Country                  | No obvious inconsistency                   |
| Full Name whitespace     | 3 issues                                   |
| Full Name capitalization | 3 issues                                   |
| Email format             | 13/13 passed basic validation              |
| Phone format             | 1 invalid record — C004                    |
| Address whitespace       | No issues                                  |
| City                     | `Manila` / `manila` inconsistency          |
| Created Date             | Mixed formats; 3 require explicit handling |
| Updated Date             | Mixed formats; 3 require explicit handling |
| Created By whitespace    | No issues                                  |
| Updated By whitespace    | No issues                                  |


In [30]:
df["Cust ID"].str.strip().ne(df["Cust ID"]).sum()

np.int64(0)

### 2.16 Customer ID — Whitespace

**Result:** Python detected 0 Customer ID records with leading or trailing whitespace.

**Finding:** No leading or trailing whitespace issues were identified in the Customer ID values.

**ETL Decision:** No Customer ID whitespace cleaning is required based on the current profiling results.

In [31]:
df["Cust ID"].str.match(r"^C\d{3}$", na=False).sum()

np.int64(16)

### 2.17 Customer ID — Format

**Result:** All 16 Customer IDs passed the expected `C` followed by 3 digits format.

**Finding:** No Customer ID format violations were identified.

**ETL Decision:** No Customer ID format transformation is required. The existing format will be preserved during ETL.

In [32]:
df[~df["Status"].isin(["Active", "Inactive"])]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
11,C011,mary ann dela cruz,mary@email.com,+639191234567,Pasig,manila,Philippines,active,2026-07-10,crm_admin,2026-07-15,crm_admin
12,C012,Robert Ong,robert@email.com,+971558887776,JVC,Dubai,UAE,ACTIVE,2026-07-11,crm_admin,2026-07-15,crm_admin


### 2.18.1 Status — Business Rule Validation

**Result:** 2 records failed the approved Status-value check: `C011` contains `active` and `C012` contains `ACTIVE`.

**Finding:** The records violate the Status standard because the values do not match the approved values `Active` or `Inactive` exactly.

**ETL Decision:** Standardize `active` and `ACTIVE` to `Active` during the cleaning stage. Do not remove the customer records.

In [33]:
df["Cust ID"].isna().sum()

np.int64(0)

### 2.18.2 Customer ID — Required Field Validation

**Result:** Python detected 0 missing Customer IDs.

**Finding:** All records contain a Customer ID. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Customer IDs. The separate duplicate Customer ID issue will be handled during the cleaning stage.

In [34]:
df["Full Name"].isna().sum()

np.int64(0)

### 2.18.3 Customer Name — Required Field Validation

**Result:** Python detected 0 missing Customer Name values.

**Finding:** All customer records contain a Customer Name. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Customer Names. The previously identified whitespace and capitalization issues will be handled during the cleaning stage.

In [35]:
df["Country"].isna().sum()

np.int64(0)

### 2.18.4 Country — Required Field Validation

**Result:** Python detected 0 missing Country values.

**Finding:** All customer records contain a Country. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Country values.

In [36]:
df["Created Date"].isna().sum()

np.int64(0)

### 2.18.5 Created Date — Required Field Validation

**Result:** Python detected 0 missing Created Date values.

**Finding:** All customer records contain a Created Date. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Created Date values. The previously identified mixed date formats and parsing issues will be handled during the cleaning/transformation stage.

In [37]:
df["Created By"].isna().sum()

np.int64(0)

### 2.18.6 Created By — Required Field Validation

**Result:** Python detected 0 missing Created By values.

**Finding:** All customer records contain a Created By value. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Created By values.

In [38]:
df["Updated Date"].isna().sum()

np.int64(0)

### 2.18.7 Updated Date — Required Field Validation

**Result:** Python detected 0 missing Updated Date values.

**Finding:** All customer records contain an Updated Date. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Updated Date values. The previously identified mixed date formats and parsing issues will be handled during the cleaning/transformation stage.

In [39]:
df["Updated By"].isna().sum()

np.int64(0)

### 2.18.8 Updated By — Required Field Validation

**Result:** Python detected 0 missing Updated By values.

**Finding:** All customer records contain an Updated By value. The required-field rule is satisfied.

**ETL Decision:** No action is required for missing Updated By values.

In [40]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df[
    df["Email Address"].notna()
    & ~df["Email Address"].str.match(email_pattern, na=False)
]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By


### 2.18.9 Email — Business Rule Validation

**Result:** No provided Email Address values failed the basic email-format validation.

**Finding:** All 13 provided email addresses comply with the basic email-format rule. The 3 missing email values are allowed because Email Address is optional.

**ETL Decision:** No email-format correction is required. Preserve missing email values as NULL.

In [41]:
phone_pattern = r"^\+?[0-9]+$"

df[
    df["Mobile"].notna()
    & ~df["Mobile"].str.match(phone_pattern, na=False)
]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
3,C004,John Cruz,john@email.com,97150ABC123,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin


### 2.18.10 Phone — Business Rule Validation

**Result:** 1 provided Mobile value failed the basic phone-format validation. The invalid record is C004 (John Cruz) with Mobile `97150ABC123`.

**Finding:** The C004 phone value contains alphabetic characters and does not comply with the basic phone-format rule. The 1 missing Mobile value is allowed because the field is optional.

**ETL Decision:** Flag C004 for CRM/business-owner review. If a correct phone number is provided, update it; otherwise, load the invalid phone value as NULL rather than loading invalid data.

In [42]:
created_date_pattern = (
    df["Created Date"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    | df["Created Date"].str.match(r"^\d{4}/\d{2}/\d{2}$", na=False)
    | df["Created Date"].str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
)

created_date_pattern.sum()

np.int64(16)

### 2.18.11 Created Date — Business Rule Validation

**Result:** All 16 Created Date values matched one of the identified source date formats: `YYYY-MM-DD`, `YYYY/MM/DD`, or `DD-MM-YYYY`.

**Finding:** All Created Date values follow a recognizable date format, but the source uses multiple formats. The dates require standardization before loading.

**ETL Decision:** Convert all Created Date values to a consistent datetime format during the transformation stage. Confirm the source date convention before converting ambiguous values and validate the final dates afterward.

In [43]:
updated_date_pattern = (
    df["Updated Date"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    | df["Updated Date"].str.match(r"^\d{4}/\d{2}/\d{2}$", na=False)
    | df["Updated Date"].str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
)

updated_date_pattern.sum()

np.int64(16)

### 2.18.12 Updated Date — Business Rule Validation

**Result:** All 16 Updated Date values matched one of the identified source date formats: `YYYY-MM-DD`, `YYYY/MM/DD`, or `DD-MM-YYYY`.

**Finding:** All Updated Date values follow a recognizable date format, but the source uses multiple formats. The dates require standardization before loading.

**ETL Decision:** Convert all Updated Date values to a consistent datetime format during the transformation stage. Confirm the source date convention before converting ambiguous values and validate the final dates afterward.

In [44]:
df["Created By"].isna().sum()

np.int64(0)

### 2.18.13 Created By — Business Rule Validation

**Result:** 0 Created By values are missing.

**Finding:** All customer records contain a Created By value. The required-field business rule is satisfied.

**ETL Decision:** No transformation or correction is required for Created By based on the current profiling results.

In [45]:
df["Updated By"].isna().sum()

np.int64(0)

### 2.18.14 Updated By — Business Rule Validation

**Result:** 0 Updated By values are missing.

**Finding:** All customer records contain an Updated By value. The required-field business rule is satisfied.

**ETL Decision:** No transformation or correction is required for Updated By based on the current profiling results.

## 2.19 Data Quality Findings Summary

| Data Quality Issue | Field | Finding | ETL Decision |
|---|---|---|---|
| Missing optional values | Email Address | 3 missing values | Preserve as NULL |
| Missing optional values | Mobile | 1 missing value | Preserve as NULL |
| Missing optional values | Address | 1 missing value | Preserve as NULL |
| Duplicate record | Cust ID | C003 appears twice and the records are identical | Keep one record and remove the exact duplicate during cleaning |
| Inconsistent capitalization | Status | C011 = `active`, C012 = `ACTIVE` | Standardize to `Active` |
| Whitespace | Full Name | 3 records contain leading/trailing spaces | Trim leading/trailing spaces |
| Inconsistent capitalization | Full Name | 3 records have inconsistent capitalization | Apply the approved business naming rule during cleaning |
| Invalid phone format | Mobile | C004 contains `97150ABC123` | Flag for CRM/business-owner review; do not invent a replacement |
| Inconsistent capitalization | City | `Manila` and `manila` both exist | Standardize capitalization during cleaning |
| Mixed date formats | Created Date | Multiple date formats identified | Convert to standardized datetime after confirming source date convention |
| Mixed date formats | Updated Date | Multiple date formats identified | Convert to standardized datetime after confirming source date convention |

## 2.20 Cleaning & Transformation Plan

| Issue | Field | Action Type | Planned Action | Requires Business Review? |
|---|---|---|---|---|
| Exact duplicate | Cust ID | Remove | Keep one C003 record and remove the exact duplicate | No |
| Whitespace | Full Name | Auto-clean | Remove leading/trailing spaces | No |
| Capitalization | Full Name | Standardize | Apply the approved customer-name formatting rule | Yes — confirm naming standard |
| Capitalization | Status | Standardize | Convert `active` and `ACTIVE` to `Active` | No |
| Capitalization | City | Standardize | Standardize `manila` to `Manila` | No |
| Mixed date formats | Created Date | Transform | Convert all dates to standardized datetime format | Yes — confirm source date convention |
| Mixed date formats | Updated Date | Transform | Convert all dates to standardized datetime format | Yes — confirm source date convention |
| Invalid phone | Mobile | Business Review | Do not invent or automatically replace the value | Yes — CRM/business owner must provide the correct value |
| Missing optional value | Email Address | Preserve | Keep missing values as NULL | No |
| Missing optional value | Mobile | Preserve | Keep missing values as NULL | No |
| Missing optional value | Address | Preserve | Keep missing values as NULL | No |

## 2.21 Business Decisions / Assumptions

| Issue | Business Question | Project Decision / Assumption | ETL Treatment |
|---|---|---|---|
| Full Name capitalization | Should customer names be standardized? | For this project, customer names will be standardized using Title Case after removing leading/trailing spaces. | Apply `.str.strip().str.title()` |
| Created Date format | What does the source date format `DD-MM-YYYY` represent? | For this project, `DD-MM-YYYY` is interpreted as Day-Month-Year. | Convert all Created Date values to standardized datetime |
| Updated Date format | What does the source date format `DD-MM-YYYY` represent? | For this project, `DD-MM-YYYY` is interpreted as Day-Month-Year. | Convert all Updated Date values to standardized datetime |
| Invalid phone — C004 | What should happen when the CRM provides an invalid phone number and no corrected value is available? | Do not invent a replacement. If the correct value cannot be confirmed, treat the invalid phone as missing. | Replace invalid phone value with NULL |

In [46]:
df["Full Name"] = df["Full Name"].str.strip()

In [47]:
df["Full Name"].str.match(r"^\s|\s$", na=False).sum()

np.int64(0)

### 2.22 Full Name — Whitespace Cleaning

**Transformation:** Removed leading and trailing whitespace from the Full Name field using `.str.strip()`.

**Validation Result:** 0 records contain remaining leading or trailing whitespace.

**ETL Decision:** Whitespace issue successfully resolved. No further whitespace cleaning is required for Full Name.

In [48]:
df["Full Name"] = df["Full Name"].str.title()

In [49]:
df["Full Name"].tolist()

['Jeff Sanchez',
 'Maria Santos',
 'Ahmed Ali',
 'John Cruz',
 'Anne Reyes',
 'Peter Lim',
 'Lara Dizon',
 'Kevin Tan',
 'Fatima Noor',
 'Chris Lee',
 'Ahmed Ali',
 'Mary Ann Dela Cruz',
 'Robert Ong',
 'Grace Yu',
 'Nina Lopez',
 'Omar Hassan']

In [50]:
df["Full Name"].str.match(r"^[A-Z][a-z]+(?: [A-Z][a-z]+)*$", na=False).sum()

np.int64(16)

### 2.23 Full Name — Capitalization Standardization

**Transformation:** Standardized Full Name capitalization using Title Case in accordance with the documented project business rule.

**Validation Result:** 16 out of 16 Full Name values passed the Title Case validation.

**ETL Decision:** Full Name capitalization has been standardized successfully. No further capitalization transformation is required under the current project business rule.

In [51]:
df["Status"] = df["Status"].str.strip().str.title()

In [52]:
df["Status"].unique()

array(['Active', 'Inactive'], dtype=object)

In [53]:
df["Status"].isin(["Active", "Inactive"]).sum()

np.int64(16)

### 2.24 Status — Standardization

**Transformation:** Removed leading/trailing whitespace and standardized Status capitalization using `.str.strip().str.title()`.

**Validation Result:** 16 out of 16 Status values match the approved values: `Active` or `Inactive`.

**ETL Decision:** Status values have been standardized successfully. No further Status transformation is required under the current project business rule.

In [54]:
df["City"].value_counts(dropna=False)

City
Dubai        7
Manila       2
Cebu         1
Singapore    1
Abu Dhabi    1
Seoul        1
manila       1
Davao        1
Sharjah      1
Name: count, dtype: int64

In [55]:
df["City"] = df["City"].str.strip().str.title()

In [56]:
df["City"].value_counts(dropna=False)

City
Dubai        7
Manila       3
Cebu         1
Singapore    1
Abu Dhabi    1
Seoul        1
Davao        1
Sharjah      1
Name: count, dtype: int64

In [57]:
df["City"].str.match(r"^[A-Z][a-z]+(?: [A-Z][a-z]+)*$", na=False).sum()

np.int64(16)

### 2.25 City — Standardization

**Transformation:** Removed leading/trailing whitespace and standardized City capitalization using `.str.strip().str.title()`.

**Validation Result:** 16 out of 16 City values passed the standardized capitalization validation.

**Finding:** The original `manila` value was standardized to `Manila`, eliminating the capitalization inconsistency.

**ETL Decision:** City values have been standardized successfully. No further City transformation is required under the current project business rule.

In [58]:
df[df["Cust ID"].duplicated(keep=False)].sort_values("Cust ID")

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
2,C003,Ahmed Ali,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,01-07-2026,crm_admin,15-07-2026,crm_admin
10,C003,Ahmed Ali,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,01-07-2026,crm_admin,15-07-2026,crm_admin


In [59]:
df[df["Cust ID"] == "C003"].duplicated(keep=False)

2     True
10    True
dtype: bool

In [60]:
df = df.drop_duplicates().reset_index(drop=True)

In [61]:
df.shape

(15, 12)

In [62]:
df["Cust ID"].duplicated().sum()

np.int64(0)

### 2.26 Duplicate Records — Removal

**Finding:** Cust ID `C003` appeared twice and both records were confirmed to be exact duplicates across all columns.

**Transformation:** Removed exact duplicate rows using `.drop_duplicates()` and reset the DataFrame index.

**Validation Result:** Dataset reduced from 16 rows to 15 rows, and 0 duplicate Cust ID values remain.

**ETL Decision:** The exact duplicate was safely removed. No further duplicate-record removal is required based on the current profiling results.

In [63]:
phone_pattern = r"^\+?[0-9]+$"

df[
    df["Mobile"].notna()
    & ~df["Mobile"].str.match(phone_pattern, na=False)
]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
3,C004,John Cruz,john@email.com,97150ABC123,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin


In [64]:
df.loc[
    df["Mobile"].notna()
    & ~df["Mobile"].str.match(phone_pattern, na=False),
    "Mobile"
] = pd.NA

In [65]:
df.loc[df["Cust ID"] == "C004", ["Cust ID", "Full Name", "Mobile"]]

,Cust ID,Full Name,Mobile
3,C004,John Cruz,<NA>


In [66]:
phone_pattern = r"^\+?[0-9]+$"

df[
    df["Mobile"].notna()
    & ~df["Mobile"].str.match(phone_pattern, na=False)
]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By


In [67]:
df["Mobile"].isna().sum()

np.int64(2)

### 2.27 Mobile — Invalid Phone Handling

**Finding:** Customer C004 contained an invalid Mobile value (`97150ABC123`) because the value contained alphabetic characters.

**Transformation:** Replaced the invalid Mobile value with `NULL` because the correct phone number could not be confirmed. The customer record itself was retained.

**Validation Result:** No non-missing Mobile values remain that violate the approved phone-format rule. The dataset now contains 2 missing Mobile values: 1 originally missing value and 1 value intentionally converted to NULL due to invalid format.

**ETL Decision:** Invalid and unverified phone values are not replaced with invented values. They are represented as NULL until a verified value is provided by the CRM/business owner.

### 2.28 Created Date — Format, Ambiguity & Validity Profiling

Before converting Created Date from text to datetime, the source values will be profiled to identify:

- Date format structure
- Potentially ambiguous dates
- Invalid calendar dates
- Unknown/unrecognized formats

No values will be modified during this profiling stage.

In [68]:
def profile_date_value(value):
    if pd.isna(value):
        return pd.Series({
            "Detected Format": "Missing",
            "Ambiguous": False,
            "Valid Date": False,
            "Reason": "Missing value"
        })

    value = str(value).strip()

    # YYYY-MM-DD or YYYY/MM/DD
    match = re.fullmatch(r"(\d{4})([-/])(\d{2})\2(\d{2})", value)

    if match:
        year, separator, month, day = match.groups()
        date_format = f"YYYY{separator}MM{separator}DD"

        try:
            pd.to_datetime(
                value,
                format=f"%Y{separator}%m{separator}%d"
            )
            valid = True
        except ValueError:
            valid = False

        return pd.Series({
            "Detected Format": date_format,
            "Ambiguous": False,
            "Valid Date": valid,
            "Reason": "Year-first format"
        })

    # DD-MM-YYYY or DD/MM/YYYY
    match = re.fullmatch(r"(\d{2})([-/])(\d{2})\2(\d{4})", value)

    if match:
        first, separator, second, year = match.groups()

        first_num = int(first)
        second_num = int(second)

        date_format = f"DD{separator}MM{separator}YYYY"

        ambiguous = first_num <= 12 and second_num <= 12

        try:
            pd.to_datetime(
                value,
                format=f"%d{separator}%m{separator}%Y"
            )
            valid = True
        except ValueError:
            valid = False

        if ambiguous:
            reason = (
                "Could be DD-MM-YYYY or MM-DD-YYYY; "
                "business rule required"
            )
        elif valid:
            reason = "Day cannot be interpreted as a month"
        else:
            reason = "Invalid calendar date"

        return pd.Series({
            "Detected Format": date_format,
            "Ambiguous": ambiguous,
            "Valid Date": valid,
            "Reason": reason
        })

    return pd.Series({
        "Detected Format": "Unknown",
        "Ambiguous": False,
        "Valid Date": False,
        "Reason": "Unrecognized date structure"
    })


date_profile = df["Created Date"].apply(profile_date_value)

date_profile

,Detected Format,Ambiguous,Valid Date,Reason
0,YYYY-MM-DD,False,True,Year-first format
1,YYYY/MM/DD,False,True,Year-first format
2,DD-MM-YYYY,True,True,Could be DD-MM-YYYY or MM-DD-YYYY; business ru...
3,YYYY-MM-DD,False,True,Year-first format
4,YYYY-MM-DD,False,True,Year-first format
5,YYYY-MM-DD,False,True,Year-first format
6,YYYY-MM-DD,False,True,Year-first format
7,YYYY-MM-DD,False,True,Year-first format
8,YYYY-MM-DD,False,True,Year-first format
9,YYYY-MM-DD,False,True,Year-first format


In [69]:
date_format_summary = (
    date_profile["Detected Format"]
    .value_counts()
    .rename_axis("Detected Format")
    .reset_index(name="Count")
)

date_format_summary["Percentage"] = (
    date_format_summary["Count"] / len(df) * 100
).round(2)

date_format_summary

,Detected Format,Count,Percentage
0,YYYY-MM-DD,13,86.67
1,YYYY/MM/DD,1,6.67
2,DD-MM-YYYY,1,6.67


In [70]:
ambiguity_summary = (
    date_profile["Ambiguous"]
    .value_counts()
    .rename_axis("Ambiguous")
    .reset_index(name="Count")
)

ambiguity_summary["Percentage"] = (
    ambiguity_summary["Count"] / len(df) * 100
).round(2)

ambiguity_summary

,Ambiguous,Count,Percentage
0,False,14,93.33
1,True,1,6.67


In [71]:
df.loc[
    date_profile["Ambiguous"],
    ["Cust ID", "Created Date"]
]

,Cust ID,Created Date
2,C003,01-07-2026


### 2.29 Created Date — Business Rule Resolution

**Profiling Finding:** Created Date contains three source formats:
- `YYYY-MM-DD` — 13 records
- `YYYY/MM/DD` — 1 record
- `DD-MM-YYYY` — 1 record

**Ambiguity Finding:** One value, `C003 = 01-07-2026`, was identified as potentially ambiguous because it could be interpreted as either `DD-MM-YYYY` or `MM-DD-YYYY`.

**Business Rule:** Dates written with the day first are interpreted using `DD-MM-YYYY`.

**Decision:** `01-07-2026` is interpreted as **1 July 2026**.

**ETL Decision:** Created Date will be converted using explicit, format-aware parsing rather than relying on automatic date inference.

### 2.30 Created Date — Explicit Date Transformation

The Created Date column contains three confirmed source formats:

- `YYYY-MM-DD`
- `YYYY/MM/DD`
- `DD-MM-YYYY`

Based on the business rule established in Section 2.29, `DD-MM-YYYY` values are interpreted as Day-Month-Year.

The values will now be converted to a standardized Pandas datetime format using explicit parsing rules rather than automatic date inference.

In [72]:
def convert_created_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return pd.to_datetime(value, format="%Y-%m-%d")

    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", value):
        return pd.to_datetime(value, format="%Y/%m/%d")

    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", value):
        return pd.to_datetime(value, format="%d-%m-%Y")

    return pd.NaT

In [73]:
df["Created Date"] = df["Created Date"].apply(convert_created_date)

In [74]:
df["Created Date"].dtype

dtype('<M8[ns]')

In [75]:
df["Created Date"].isna().sum()

np.int64(0)

In [76]:
df.loc[
    df["Cust ID"] == "C003",
    ["Cust ID", "Created Date"]
]

,Cust ID,Created Date
2,C003,2026-07-01


### 2.30 Created Date — Explicit Date Transformation

**Transformation:** Converted the Created Date column from text/object to a standardized Pandas datetime datatype using explicit format-aware parsing.

**Formats Processed:**
- `YYYY-MM-DD`
- `YYYY/MM/DD`
- `DD-MM-YYYY`

**Business Rule Applied:** `DD-MM-YYYY` values are interpreted as Day-Month-Year. Therefore, `C003 = 01-07-2026` was converted to `2026-07-01`.

**Validation Result:**
- Data type: `datetime64[ns]`
- Failed conversions / `NaT`: 0
- Business-rule test: C003 successfully converted to `2026-07-01`

**ETL Decision:** Automatic date inference was avoided. Known source formats and documented business rules were explicitly applied before conversion.

In [77]:
df["Created Date"].agg(["min", "max"])

min   2026-07-01
max   2026-07-14
Name: Created Date, dtype: datetime64[ns]

In [78]:
df[
    (df["Created Date"] < "2026-07-01") |
    (df["Created Date"] > "2026-07-14")
][["Cust ID", "Created Date"]]

,Cust ID,Created Date


In [79]:
df[["Cust ID", "Created Date"]].sort_values("Created Date")

,Cust ID,Created Date
0,C001,2026-07-01
2,C003,2026-07-01
1,C002,2026-07-02
3,C004,2026-07-03
4,C005,2026-07-04
5,C006,2026-07-05
6,C007,2026-07-06
7,C008,2026-07-07
8,C009,2026-07-08
9,C010,2026-07-09


### 2.31 Created Date — Final Validation

**Validation Results:**
- Data type: `datetime64[ns]`
- Failed conversions / `NaT`: 0
- Minimum date: `2026-07-01`
- Maximum date: `2026-07-14`
- Out-of-range records: 0
- Full-column inspection: All 15 records successfully converted and verified
- Business-rule validation: `C003 = 01-07-2026` was correctly converted to `2026-07-01`

**Conclusion:** Created Date has been successfully standardized, converted to datetime, and validated. No invalid, failed, or out-of-range dates were identified.

### 2.32 Updated Date — Format, Ambiguity & Validity Profiling

Before converting Updated Date from text/object to datetime, the source values will be profiled to identify:

- Date format structure
- Potentially ambiguous dates
- Invalid calendar dates
- Unknown/unrecognized formats

No values will be modified during this profiling stage.

In [80]:
updated_date_profile = df["Updated Date"].apply(profile_date_value)

updated_date_profile

,Detected Format,Ambiguous,Valid Date,Reason
0,YYYY-MM-DD,False,True,Year-first format
1,YYYY/MM/DD,False,True,Year-first format
2,DD-MM-YYYY,False,True,Day cannot be interpreted as a month
3,YYYY-MM-DD,False,True,Year-first format
4,YYYY-MM-DD,False,True,Year-first format
5,YYYY-MM-DD,False,True,Year-first format
6,YYYY-MM-DD,False,True,Year-first format
7,YYYY-MM-DD,False,True,Year-first format
8,YYYY-MM-DD,False,True,Year-first format
9,YYYY-MM-DD,False,True,Year-first format


In [81]:
updated_date_format_summary = (
    updated_date_profile["Detected Format"]
    .value_counts()
    .rename_axis("Detected Format")
    .reset_index(name="Count")
)

updated_date_format_summary["Percentage"] = (
    updated_date_format_summary["Count"] / len(df) * 100
).round(2)

updated_date_format_summary

,Detected Format,Count,Percentage
0,YYYY-MM-DD,13,86.67
1,YYYY/MM/DD,1,6.67
2,DD-MM-YYYY,1,6.67


In [82]:
updated_ambiguity_summary = (
    updated_date_profile["Ambiguous"]
    .value_counts()
    .rename_axis("Ambiguous")
    .reset_index(name="Count")
)

updated_ambiguity_summary["Percentage"] = (
    updated_ambiguity_summary["Count"] / len(df) * 100
).round(2)

updated_ambiguity_summary

,Ambiguous,Count,Percentage
0,False,15,100.0


In [83]:
df.loc[
    updated_date_profile["Detected Format"] == "DD-MM-YYYY",
    ["Cust ID", "Updated Date"]
]

,Cust ID,Updated Date
2,C003,15-07-2026


### 2.33 Updated Date — Business Rule Resolution

**Profiling Finding:** Updated Date contains three source formats:
- `YYYY-MM-DD` — 13 records
- `YYYY/MM/DD` — 1 record
- `DD-MM-YYYY` — 1 record

**Ambiguity Finding:** No Updated Date values were identified as ambiguous.

**Non-standard Format:** `C003 = 15-07-2026` was identified as `DD-MM-YYYY`. This value is unambiguous because `15` cannot represent a month.

**Decision:** `15-07-2026` is interpreted as **15 July 2026**.

**ETL Decision:** Updated Date will be converted using explicit, format-aware parsing rather than automatic date inference.

In [84]:
def convert_updated_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return pd.to_datetime(value, format="%Y-%m-%d")

    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", value):
        return pd.to_datetime(value, format="%Y/%m/%d")

    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", value):
        return pd.to_datetime(value, format="%d-%m-%Y")

    return pd.NaT

In [85]:
df["Updated Date"] = df["Updated Date"].apply(convert_updated_date)

In [86]:
df["Updated Date"].dtype

dtype('<M8[ns]')

In [87]:
df["Updated Date"].isna().sum()

np.int64(0)

In [88]:
df.loc[
    df["Cust ID"] == "C003",
    ["Cust ID", "Updated Date"]
]

,Cust ID,Updated Date
2,C003,2026-07-15


In [89]:
df["Updated Date"].agg(["min", "max"])

min   2026-07-10
max   2026-07-15
Name: Updated Date, dtype: datetime64[ns]

In [90]:
df[
    (df["Updated Date"] < "2026-07-01") |
    (df["Updated Date"] > "2026-07-15")
][["Cust ID", "Updated Date"]]

,Cust ID,Updated Date


In [91]:
df[["Cust ID", "Updated Date"]].sort_values("Updated Date")

,Cust ID,Updated Date
1,C002,2026-07-10
3,C004,2026-07-11
4,C005,2026-07-12
5,C006,2026-07-13
6,C007,2026-07-14
0,C001,2026-07-15
2,C003,2026-07-15
7,C008,2026-07-15
8,C009,2026-07-15
9,C010,2026-07-15


### 2.35 Updated Date — Final Validation

**Validation Results:**
- Data type: `datetime64[ns]`
- Failed conversions / `NaT`: 0
- Minimum date: `2026-07-10`
- Maximum date: `2026-07-15`
- Out-of-range records: 0
- C003 validation: `15-07-2026` successfully converted to `2026-07-15`

**Conclusion:** Updated Date has been successfully standardized, converted to datetime, and validated. No invalid or out-of-range dates were identified.

### 2.36 Final Customer Dataset Validation

After completing the individual cleaning and transformation steps, the complete customer dataset will undergo a final data-quality validation before PostgreSQL loading.

The validation will confirm:

- Expected row count
- Customer ID uniqueness
- Missing-value status
- Final column data types
- Standardized categorical values
- Phone number handling
- Date column readiness
- Overall dataset structure

No data will be modified during this validation stage.

In [92]:
df.shape

(15, 12)

In [93]:
df["Cust ID"].nunique()

15

In [94]:
df.isna().sum()

Cust ID          0
Full Name        0
Email Address    2
Mobile           2
Address          1
City             0
Country          0
Status           0
Created Date     0
Created By       0
Updated Date     0
Updated By       0
dtype: int64

In [95]:
df[df.isna().any(axis=1)]

,Cust ID,Full Name,Email Address,Mobile,Address,City,Country,Status,Created Date,Created By,Updated Date,Updated By
2,C003,Ahmed Ali,NaN,+971501112223,Al Nahda,Dubai,UAE,Active,2026-07-01,crm_admin,2026-07-15,crm_admin
3,C004,John Cruz,john@email.com,<NA>,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin
4,C005,Anne Reyes,anne@email.com,+639181112223,NaN,Cebu,Philippines,Active,2026-07-04,crm_admin,2026-07-12,crm_admin
6,C007,Lara Dizon,lara@email.com,NaN,Quezon City,Manila,Philippines,Active,2026-07-06,crm_admin,2026-07-14,crm_admin
13,C014,Nina Lopez,NaN,+639177778888,Davao,Davao,Philippines,Active,2026-07-13,crm_admin,2026-07-15,crm_admin


In [96]:
df.dtypes

Cust ID                  object
Full Name                object
Email Address            object
Mobile                   object
Address                  object
City                     object
Country                  object
Status                   object
Created Date     datetime64[ns]
Created By               object
Updated Date     datetime64[ns]
Updated By               object
dtype: object

In [97]:
print("Status:")
print(df["Status"].unique())

print("\nCity:")
print(df["City"].unique())

Status:
['Active' 'Inactive']

City:
['Dubai' 'Manila' 'Cebu' 'Singapore' 'Abu Dhabi' 'Seoul' 'Davao' 'Sharjah']


In [98]:
df[["Cust ID", "Mobile"]]

,Cust ID,Mobile
0,C001,+971559522199
1,C002,+639171234567
2,C003,+971501112223
3,C004,<NA>
4,C005,+639181112223
5,C006,+6591234567
6,C007,NaN
7,C008,+971556667778
8,C009,+971509998887
9,C010,+821012345678


In [99]:
df["Mobile"].notna().sum()

np.int64(13)

In [100]:
df[["Created Date", "Updated Date"]].isna().sum()

Created Date    0
Updated Date    0
dtype: int64

In [101]:
df[["Cust ID", "Email Address", "Mobile", "Address"]].isna().sum()

Cust ID          0
Email Address    2
Mobile           2
Address          1
dtype: int64

In [102]:
df.duplicated().sum()

np.int64(0)

In [103]:
df.columns.tolist()

['Cust ID',
 'Full Name',
 'Email Address',
 'Mobile',
 'Address',
 'City',
 'Country',
 'Status',
 'Created Date',
 'Created By',
 'Updated Date',
 'Updated By']

### 2.36 Final Customer Dataset Validation

**Final Dataset Structure:**
- Rows: 15
- Columns: 12
- Customer IDs: 15 unique values
- Complete duplicate rows: 0

**Data Types:**
- Created Date: `datetime64[ns]`
- Updated Date: `datetime64[ns]`
- Other fields remain text/object as expected.

**Data Quality:**
- Created Date missing values: 0
- Updated Date missing values: 0
- Mobile values present: 13
- Missing Email Address: 2
- Missing Mobile: 2
- Missing Address: 1

**Standardization:**
- Status values standardized to `Active` and `Inactive`.
- City values standardized with consistent capitalization.
- Date formats standardized and validated.
- Invalid Mobile value was converted to missing rather than retained as invalid data.

**Conclusion:** The customer dataset has completed profiling, cleaning, transformation, and final data-quality validation. The dataset is ready for the next ETL stage: preparation for PostgreSQL loading.

In [104]:
import psycopg2

In [105]:
# Run this once if psycopg2 is not installed in the current environment
# %pip install psycopg2-binary

### 2.37 PostgreSQL Connection & Target Schema Inspection

A connection to the PostgreSQL database `lj_dev_commerce` is established from the notebook.

The PostgreSQL connection is established before target-schema mapping so that the actual target table structure can be inspected and used as the source of truth for the remaining transformation steps.

**Target Table:** `commerce.customer`

The target schema is inspected directly through PostgreSQL and contains 12 columns covering:

- Customer identification
- Customer contact information
- Location information
- Customer status
- Creation audit fields
- Update audit fields

No data is loaded or modified during this stage.

In [106]:
from getpass import getpass

password = getpass("PostgreSQL password: ")

In [107]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="lj_dev_commerce",
    user="postgres",
    password=password
)

In [108]:
conn.status

1

In [109]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'commerce'
      AND table_name = 'customer'
    ORDER BY ordinal_position;
""")

target_schema = cursor.fetchall()

In [110]:
target_schema

[(1, 'customer_id', 'text'),
 (2, 'customer_name', 'text'),
 (3, 'email', 'text'),
 (4, 'phone', 'text'),
 (5, 'address', 'text'),
 (6, 'city', 'text'),
 (7, 'country', 'text'),
 (8, 'customer_status', 'text'),
 (9, 'created_date', 'timestamp without time zone'),
 (10, 'created_by', 'text'),
 (11, 'updated_date', 'timestamp without time zone'),
 (12, 'updated_by', 'text')]

### 2.38 Source-to-Target Column Mapping

The source CRM column names are mapped to the actual PostgreSQL target column names identified during schema inspection.

The mapping ensures that each source field is aligned with the corresponding target field before the DataFrame is renamed and prepared for loading.

No data values are modified during this step; only the column-name mapping is defined.

In [111]:
column_mapping = {
    "Cust ID": "customer_id",
    "Full Name": "customer_name",
    "Email Address": "email",
    "Mobile": "phone",
    "Address": "address",
    "City": "city",
    "Country": "country",
    "Status": "customer_status",
    "Created Date": "created_date",
    "Created By": "created_by",
    "Updated Date": "updated_date",
    "Updated By": "updated_by"
}

In [112]:
column_mapping

{'Cust ID': 'customer_id',
 'Full Name': 'customer_name',
 'Email Address': 'email',
 'Mobile': 'phone',
 'Address': 'address',
 'City': 'city',
 'Country': 'country',
 'Status': 'customer_status',
 'Created Date': 'created_date',
 'Created By': 'created_by',
 'Updated Date': 'updated_date',
 'Updated By': 'updated_by'}

### 2.39 Apply Source-to-Target Column Mapping

The validated customer DataFrame is renamed using the source-to-target mapping defined in the previous step.

This aligns the DataFrame column names with the PostgreSQL target schema while preserving the underlying data values.

No row-level data transformation is performed during this step.

In [113]:
df = df.rename(columns=column_mapping)

In [114]:
df.columns.tolist()

['customer_id',
 'customer_name',
 'email',
 'phone',
 'address',
 'city',
 'country',
 'customer_status',
 'created_date',
 'created_by',
 'updated_date',
 'updated_by']

### 2.40 Validate Target Column Structure

The renamed DataFrame is validated against the PostgreSQL target schema retrieved during the schema inspection stage.

This validation confirms that:

- All expected target columns are present.
- No unexpected columns were introduced.
- The DataFrame column order matches the PostgreSQL target table.
- The row count remains unchanged after column renaming.

No data is loaded into PostgreSQL during this validation.

In [115]:
expected_columns = [column[1] for column in target_schema]

In [116]:
df.columns.tolist() == expected_columns

True

In [117]:
df.shape

(15, 12)

### 2.41 Validate Data Types Against PostgreSQL

The DataFrame data types are validated against the data types defined in the PostgreSQL target schema.

Text-based fields are expected to remain compatible with PostgreSQL `text` columns, while the Created Date and Updated Date fields are expected to be represented as datetime values compatible with PostgreSQL timestamp columns.

No data is loaded or modified during this validation.

In [118]:
df.dtypes

customer_id                object
customer_name              object
email                      object
phone                      object
address                    object
city                       object
country                    object
customer_status            object
created_date       datetime64[ns]
created_by                 object
updated_date       datetime64[ns]
updated_by                 object
dtype: object

### 2.42 Validate PostgreSQL Nullability

The PostgreSQL target table is inspected to determine which columns allow NULL values and whether any columns have database-level default values.

This validation ensures that the cleaned DataFrame is compatible with the target table's nullability requirements before data loading.

No data is loaded or modified during this validation.

In [119]:
target_nullability = cursor.execute("""
    SELECT
        ordinal_position,
        column_name,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'commerce'
      AND table_name = 'customer'
    ORDER BY ordinal_position;
""")

target_nullability = cursor.fetchall()

In [120]:
target_nullability

[(1, 'customer_id', 'NO', None),
 (2, 'customer_name', 'NO', None),
 (3, 'email', 'YES', None),
 (4, 'phone', 'YES', None),
 (5, 'address', 'YES', None),
 (6, 'city', 'YES', None),
 (7, 'country', 'YES', None),
 (8, 'customer_status', 'NO', None),
 (9, 'created_date', 'NO', None),
 (10, 'created_by', 'NO', None),
 (11, 'updated_date', 'NO', None),
 (12, 'updated_by', 'NO', None)]

### 2.43 Validate NOT NULL Fields

The DataFrame is checked against the PostgreSQL columns that do not allow NULL values.

This validation confirms that all required target fields contain valid values before the dataset is loaded into PostgreSQL.

No data is loaded or modified during this validation.

In [121]:
required_columns = [
    column[1]
    for column in target_nullability
    if column[2] == "NO"
]

In [122]:
df[required_columns].isna().sum()

customer_id        0
customer_name      0
customer_status    0
created_date       0
created_by         0
updated_date       0
updated_by         0
dtype: int64

### 2.44 Validate Primary Key / Unique Constraint

The PostgreSQL target table is inspected to identify primary key and unique constraints.

This validation determines whether `customer_id` must be unique in the target table and helps prevent duplicate-key errors during data loading.

No data is loaded or modified during this validation.

In [123]:
constraints = cursor.execute("""
    SELECT
        tc.constraint_name,
        tc.constraint_type,
        kcu.column_name
    FROM information_schema.table_constraints AS tc
    JOIN information_schema.key_column_usage AS kcu
        ON tc.constraint_name = kcu.constraint_name
        AND tc.table_schema = kcu.table_schema
        AND tc.table_name = kcu.table_name
    WHERE tc.table_schema = 'commerce'
      AND tc.table_name = 'customer'
      AND tc.constraint_type IN ('PRIMARY KEY', 'UNIQUE')
    ORDER BY tc.constraint_type, kcu.ordinal_position;
""")

constraints = cursor.fetchall()

In [124]:
constraints

[('customer_pkey', 'PRIMARY KEY', 'customer_id')]

In [125]:
df["customer_id"].nunique() == len(df)

True

### 2.45 Final Pre-Load Validation

The final DataFrame is validated against the PostgreSQL target requirements before data loading.

This validation confirms:

- Expected row count
- Expected column count
- Target column names and order
- Customer ID uniqueness
- Completeness of PostgreSQL NOT NULL fields
- Readiness of the Created Date and Updated Date fields

No data is loaded or modified in PostgreSQL during this validation.

In [126]:
print("Row count:", len(df))
print("Column count:", len(df.columns))
print("Columns match target:", df.columns.tolist() == expected_columns)
print("Customer IDs unique:", df["customer_id"].nunique() == len(df))
print("Required fields complete:", df[required_columns].isna().sum().sum() == 0)
print("Created Date NaT:", df["created_date"].isna().sum())
print("Updated Date NaT:", df["updated_date"].isna().sum())

Row count: 15
Column count: 12
Columns match target: True
Customer IDs unique: True
Required fields complete: True
Created Date NaT: 0
Updated Date NaT: 0


### 2.46 Prepare PostgreSQL Insert

The validated DataFrame is prepared for insertion into the PostgreSQL target table.

The target column order is taken from the PostgreSQL schema, and DataFrame missing values are converted to Python `None` so they can be inserted as SQL NULL values.

A parameterized INSERT statement is prepared for the `commerce.customer` table.

The INSERT statement is not executed during this stage.

In [127]:
insert_columns = expected_columns

insert_records = [
    tuple(None if pd.isna(value) else value for value in row)
    for row in df[insert_columns].itertuples(index=False, name=None)
]

In [128]:
len(insert_records), len(insert_records[0])

(15, 12)

In [129]:
placeholders = ", ".join(["%s"] * len(insert_columns))
columns_sql = ", ".join(insert_columns)

insert_sql = f"""
INSERT INTO commerce.customer ({columns_sql})
VALUES ({placeholders});
"""

In [130]:
print(insert_sql)


INSERT INTO commerce.customer (customer_id, customer_name, email, phone, address, city, country, customer_status, created_date, created_by, updated_date, updated_by)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);



### 2.47 Load Data into PostgreSQL

The validated customer records are inserted into the PostgreSQL target table `commerce.customer` using the prepared parameterized INSERT statement.

The prepared records are inserted using the existing PostgreSQL connection and transaction.

The transaction is committed only after the INSERT operation completes successfully.

In [131]:
cursor.executemany(insert_sql, insert_records)
conn.commit()

print(f"{len(insert_records)} records inserted successfully.")

15 records inserted successfully.


### 2.48 Verify PostgreSQL Row Count

The PostgreSQL target table is queried after the data load to confirm the number of records currently stored in `commerce.customer`.

The loaded PostgreSQL row count is compared with the expected DataFrame row count to verify that all prepared customer records were inserted successfully.

In [132]:
target_row_count = cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.customer;
""")

target_row_count = cursor.fetchone()[0]

print("PostgreSQL row count:", target_row_count)
print("Expected row count:", len(df))
print("Row count matches:", target_row_count == len(df))

PostgreSQL row count: 15
Expected row count: 15
Row count matches: True


### 2.49 Verify Loaded Data

The records stored in the PostgreSQL target table are retrieved and compared with the final validated DataFrame.

This validation confirms that the loaded customer records match the prepared source data after the PostgreSQL load.

In [133]:
loaded_df = pd.read_sql_query(
    """
    SELECT
        customer_id,
        customer_name,
        email,
        phone,
        address,
        city,
        country,
        customer_status,
        created_date,
        created_by,
        updated_date,
        updated_by
    FROM commerce.customer
    ORDER BY customer_id;
    """,
    conn
)

C:\Users\Jeff\AppData\Local\Temp\ipykernel_18100\1708353168.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  loaded_df = pd.read_sql_query(


In [134]:
loaded_df.shape

(15, 12)

The loaded PostgreSQL data is compared with the final validated DataFrame after aligning the records by `customer_id`.

Missing values are treated as equivalent when both datasets contain a missing value.

The comparison counts actual cell-value differences to confirm that the loaded data matches the prepared data.

In [142]:
source_for_comparison = df[expected_columns].sort_values("customer_id").reset_index(drop=True)

loaded_for_comparison = loaded_df[expected_columns].sort_values("customer_id").reset_index(drop=True)

comparison_mask = (
    (source_for_comparison == loaded_for_comparison)
    | (source_for_comparison.isna() & loaded_for_comparison.isna())
)

actual_differences = comparison_mask.eq(False).sum().sum()

print("Actual cell-value differences:", actual_differences)
print("Loaded data matches prepared data:", actual_differences == 0)

Actual cell-value differences: 0
Loaded data matches prepared data: True


### 2.50 PostgreSQL Load Completion

The customer data load has been completed and verified in the PostgreSQL target table.

The target row count matches the expected source row count, and the loaded records contain zero actual cell-value differences from the final validated DataFrame.

The PostgreSQL transaction has been committed successfully.

No further data modifications are performed during this stage.

In [143]:
print("PostgreSQL customer load completed successfully.")
print("Records loaded:", target_row_count)
print("Data verification passed:", actual_differences == 0)

PostgreSQL customer load completed successfully.
Records loaded: 15
Data verification passed: True


### 2.51 Inspect Loaded Customer Data

The records loaded into `commerce.customer` are retrieved from PostgreSQL for a final visual inspection.

This check is used to confirm that the customer records are present in the target table and that the stored values appear as expected.

In [144]:
cursor.execute("""
    SELECT *
    FROM commerce.customer
    ORDER BY customer_id;
""")

customer_data = cursor.fetchall()

customer_data

[('C001',
  'Jeff Sanchez',
  'jeff@email.com',
  '+971559522199',
  'Al Jafiliya',
  'Dubai',
  'UAE',
  'Active',
  datetime.datetime(2026, 7, 1, 0, 0),
  'crm_admin',
  datetime.datetime(2026, 7, 15, 0, 0),
  'crm_admin'),
 ('C002',
  'Maria Santos',
  'maria@email.com',
  '+639171234567',
  'Makati',
  'Manila',
  'Philippines',
  'Active',
  datetime.datetime(2026, 7, 2, 0, 0),
  'crm_admin',
  datetime.datetime(2026, 7, 10, 0, 0),
  'crm_admin'),
 ('C003',
  'Ahmed Ali',
  None,
  '+971501112223',
  'Al Nahda',
  'Dubai',
  'UAE',
  'Active',
  datetime.datetime(2026, 7, 1, 0, 0),
  'crm_admin',
  datetime.datetime(2026, 7, 15, 0, 0),
  'crm_admin'),
 ('C004',
  'John Cruz',
  'john@email.com',
  None,
  'Business Bay',
  'Dubai',
  'UAE',
  'Inactive',
  datetime.datetime(2026, 7, 3, 0, 0),
  'crm_admin',
  datetime.datetime(2026, 7, 11, 0, 0),
  'crm_admin'),
 ('C005',
  'Anne Reyes',
  'anne@email.com',
  '+639181112223',
  None,
  'Cebu',
  'Philippines',
  'Active',
  dateti

In [145]:
pd.DataFrame(customer_data, columns=expected_columns)

,customer_id,customer_name,email,phone,address,city,country,customer_status,created_date,created_by,updated_date,updated_by
0,C001,Jeff Sanchez,jeff@email.com,+971559522199,Al Jafiliya,Dubai,UAE,Active,2026-07-01,crm_admin,2026-07-15,crm_admin
1,C002,Maria Santos,maria@email.com,+639171234567,Makati,Manila,Philippines,Active,2026-07-02,crm_admin,2026-07-10,crm_admin
2,C003,Ahmed Ali,None,+971501112223,Al Nahda,Dubai,UAE,Active,2026-07-01,crm_admin,2026-07-15,crm_admin
3,C004,John Cruz,john@email.com,None,Business Bay,Dubai,UAE,Inactive,2026-07-03,crm_admin,2026-07-11,crm_admin
4,C005,Anne Reyes,anne@email.com,+639181112223,None,Cebu,Philippines,Active,2026-07-04,crm_admin,2026-07-12,crm_admin
5,C006,Peter Lim,peter@email.com,+6591234567,Orchard,Singapore,Singapore,Active,2026-07-05,crm_admin,2026-07-13,crm_admin
6,C007,Lara Dizon,lara@email.com,None,Quezon City,Manila,Philippines,Active,2026-07-06,crm_admin,2026-07-14,crm_admin
7,C008,Kevin Tan,kevin@email.com,+971556667778,Marina,Abu Dhabi,UAE,Active,2026-07-07,crm_admin,2026-07-15,crm_admin
8,C009,Fatima Noor,fatima@email.com,+971509998887,Deira,Dubai,UAE,Active,2026-07-08,crm_admin,2026-07-15,crm_admin
9,C010,Chris Lee,chris@email.com,+821012345678,Gangnam,Seoul,Korea,Active,2026-07-09,crm_admin,2026-07-15,crm_admin
